# Week 3 Threshold Policy Lab

This 20-minute, ungraded lab uses illustrative transaction scores. Work in pairs. The model has already produced probabilities; your task is to choose an operating policy under a 400-review daily capacity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n, n_positive = 10_000, 400
actual = np.r_[np.ones(n_positive, dtype=int), np.zeros(n - n_positive, dtype=int)]
score = np.r_[rng.beta(5, 2, n_positive), rng.beta(1.4, 7, n - n_positive)]
order = rng.permutation(n)
cases = pd.DataFrame({'actual_positive': actual[order], 'score': score[order]})
cases.head()

## 1. See how a threshold becomes workload
The chart ranks cases by predicted fraud probability. Change the threshold or review capacity, then compare the selected workload with the errors the policy creates.

In [ ]:
def policy_view(data, threshold=0.55, capacity=400, display_limit=1200):
    ranked = data.sort_values('score', ascending=False).reset_index(drop=True)
    ranked['rank'] = np.arange(1, len(ranked) + 1)
    action = ranked['score'] >= threshold
    positive = ranked['actual_positive'].astype(bool)
    tp = int((action & positive).sum())
    fp = int((action & ~positive).sum())
    fn = int((~action & positive).sum())
    reviews = tp + fp

    shown = ranked.head(display_limit)
    colors = np.where(shown['actual_positive'].eq(1), '#2A62AD', '#B7C0CC')
    fig, ax = plt.subplots(figsize=(11, 4.8))
    ax.scatter(shown['rank'], shown['score'], c=colors, s=18, alpha=0.78, linewidths=0)
    ax.axhline(threshold, color='#C96F16', linewidth=2.5, label=f'threshold = {threshold:.2f}')
    ax.axvline(capacity, color='#1B7F5A', linewidth=2, linestyle='--', label=f'capacity = {capacity}')
    ax.fill_between([1, display_limit], threshold, 1, color='#C96F16', alpha=0.07)
    ax.set_xlim(1, display_limit)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel('Rank by predicted probability')
    ax.set_ylabel('Predicted fraud probability')
    ax.set_title('Ranked probabilities, action threshold, and review capacity', loc='left', weight='bold')
    ax.legend(loc='upper right', frameon=False)
    ax.grid(axis='y', color='#E1E6EE', linewidth=0.8)
    plt.show()

    return pd.DataFrame({
        'policy quantity': ['reviews selected', 'capacity', 'true positives', 'false positives', 'false negatives'],
        'count': [reviews, capacity, tp, fp, fn]
    }).set_index('policy quantity')

policy_view(cases, threshold=0.55, capacity=400)

**Try it:** rerun the cell with `threshold=0.70`, then with `threshold=0.40`. Which policy fits capacity? Which error grows as the threshold rises?

## 2. Predict before running
For each threshold below, predict the direction of workload, false positives, false negatives, precision, and recall.

In [ ]:
def threshold_metrics(data, threshold):
    action = data['score'] >= threshold
    positive = data['actual_positive'].astype(bool)
    tp = int((action & positive).sum())
    fp = int((action & ~positive).sum())
    fn = int((~action & positive).sum())
    tn = int((~action & ~positive).sum())
    return {
        'threshold': threshold, 'reviews': tp + fp, 'TP': tp, 'FP': fp,
        'FN': fn, 'TN': tn,
        'precision': tp / (tp + fp) if tp + fp else 0,
        'recall': tp / (tp + fn) if tp + fn else 0,
        'within_capacity': tp + fp <= 400
    }

thresholds = [0.25, 0.40, 0.55, 0.70]
threshold_table = pd.DataFrame(threshold_metrics(cases, t) for t in thresholds)
threshold_table.assign(precision_pct=(100 * threshold_table['precision']).round(1), recall_pct=(100 * threshold_table['recall']).round(1))[['threshold', 'reviews', 'TP', 'FP', 'FN', 'TN', 'precision_pct', 'recall_pct', 'within_capacity']]

In [ ]:
ax = threshold_table.plot(x='threshold', y=['reviews', 'FP', 'FN'], marker='o', figsize=(8, 4))
ax.axhline(400, color='black', linestyle='--', label='review capacity')
ax.set_ylabel('Cases per day')
ax.legend()
plt.show()

## 3. Compare a threshold with a top-k rule
A top-k policy fixes workload directly. Compare 150, 400, and 1,200 reviews with the threshold options.

In [ ]:
def top_k_metrics(data, k):
    selected = data.nlargest(k, 'score').index
    action = data.index.isin(selected)
    positive = data['actual_positive'].astype(bool).to_numpy()
    tp = int((action & positive).sum())
    fp = int((action & ~positive).sum())
    fn = int((~action & positive).sum())
    return {'policy': f'top {k}', 'reviews': k, 'TP': tp, 'FP': fp, 'FN': fn,
            'precision': tp / k, 'recall': tp / n_positive}

top_k_table = pd.DataFrame(top_k_metrics(cases, k) for k in [150, 400, 1200])
top_k_table.assign(precision_pct=(100 * top_k_table['precision']).round(1), recall_pct=(100 * top_k_table['recall']).round(1))[['policy', 'reviews', 'TP', 'FP', 'FN', 'precision_pct', 'recall_pct']]

## 4. Add costs and test sensitivity
Change either cost assumption. Does your preferred feasible policy change?

In [ ]:
false_positive_cost = 20
false_negative_cost = 500

threshold_costs = threshold_table.assign(policy=lambda d: d['threshold'].map(lambda t: f'threshold {t:.2f}'))
top_k_costs = top_k_table.assign(within_capacity=lambda d: d['reviews'] <= 400)
policy_cost_table = pd.concat([
    threshold_costs[['policy', 'reviews', 'FP', 'FN', 'within_capacity']],
    top_k_costs[['policy', 'reviews', 'FP', 'FN', 'within_capacity']]
], ignore_index=True)
policy_cost_table['expected_cost'] = (
    false_positive_cost * policy_cost_table['FP'] + false_negative_cost * policy_cost_table['FN']
)
policy_cost_table.sort_values(['within_capacity', 'expected_cost'], ascending=[False, True])

## 5. Participation-card handoff
Record the six card elements: (1) your operating policy, (2) one supporting count or rate, (3) the costly error, (4) the capacity rule, (5) one sensitivity result, and (6) one remaining uncertainty.